In [ ]:
!pip install -qU transformers accelerate bitsandbytes

In [ ]:
pip install PyMuPDF nltk sentence-transformers scikit-learn

In [ ]:
!pip install telebot

In [ ]:
import numpy as np
import transformers
import torch
from torch import cuda, bfloat16
import fitz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import telebot

In [ ]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

llm = transformers.AutoModelForCausalLM.from_pretrained(
    'microsoft/Phi-4-mini-instruct',
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto'
)


tokenizer = transformers.AutoTokenizer.from_pretrained('microsoft/Phi-4-mini-instruct',trust_remote_code=True)


llm.eval()

In [ ]:
def call_llm(prompt):
  inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
  outputs = llm.generate(
      **inputs,
      max_new_tokens=256,
      do_sample=True,
      temperature=0.7,
      top_p=0.95,
      use_cache=True
  )

  return tokenizer.decode(outputs[0], skip_special_tokens=True)

#Load a BOOK

In [ ]:
YOUR_PDF_PATH = '' #/content/Hands-On_Machine_Learning_with_Scikit-Learn-Keras-and-TensorFlow-2nd-Edition-Aurelien-Geron.pdf
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_text = extract_text_from_pdf(YOUR_PDF_PATH)


#Chunck it , Don't forget Overlapping

In [ ]:
def split_into_paragraphs(text):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    return paragraphs

paragraphs = split_into_paragraphs(pdf_text)

In [ ]:

model = SentenceTransformer("all-MiniLM-L6-v2")
paragraph_embeddings = model.encode(paragraphs)


In [ ]:
def semantic_overlap_chunking(paragraphs, embeddings, window_size=5, stride=3, similarity_threshold=0.7):
    chunks = []
    i = 0

    while i < len(paragraphs):
        current_paragraphs = paragraphs[i:i + window_size]
        current_embeddings = embeddings[i:i + window_size]

        avg_sim = 0
        if len(current_embeddings) > 1:
            sims = [cosine_similarity([current_embeddings[j]], [current_embeddings[j + 1]])[0][0]
                    for j in range(len(current_embeddings) - 1)]
            avg_sim = sum(sims) / len(sims)

        if avg_sim >= similarity_threshold or i + window_size >= len(paragraphs):
            chunk_text = " ".join(current_paragraphs)
            chunks.append(chunk_text)
            i += stride
        else:
            i += 1

    return chunks

semantic_chunks = semantic_overlap_chunking(paragraphs, paragraph_embeddings)


#Embeddings

In [ ]:
chunk_embeddings = model.encode(semantic_chunks, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
embedding_matrix = np.array(chunk_embeddings).astype("float32")

chunk_texts = semantic_chunks


In [ ]:
def retrieve_chunks(query, top_k=3):
    query_vec = model.encode([query]).astype("float32")

    sims = cosine_similarity(query_vec, embedding_matrix)[0]

    top_indices = sims.argsort()[::-1][:top_k]

    return [chunk_texts[i] for i in top_indices]


# Prompt Construction

In [ ]:
def build_prompt(chunks, query, max_tokens=3000):
    prompt = f"User query: {query}\n\nRelevant context:\n"
    total_tokens = 0
    included_chunks = []

    for chunk in chunks:
        chunk_tokens = len(tokenizer(chunk)["input_ids"])
        if total_tokens + chunk_tokens > max_tokens:
            break
        included_chunks.append(chunk)
        total_tokens += chunk_tokens

    context = "\n\n".join(included_chunks)
    return f"{prompt}{context}\n\nAnswer the user's question based on the above."


In [ ]:

bot_token = "YOUR_BOT_TOKEN"

bot = telebot.TeleBot(bot_token)

@bot.message_handler(func=lambda message: True)
def handle_message(message):
    chat_id = message.chat.id
    query = message.text

    chunks = retrieve_chunks(query)
    prompt = build_prompt(chunks, query)
    response = call_llm(prompt)

    bot.send_message(chat_id, response)

bot.infinity_polling()


2025-07-20 20:37:06,433 (__init__.py:1121 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
ERROR:TeleBot:Infinity polling: polling exited
2025-07-20 20:37:06,435 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling
